In [1]:
import pandas as pd
import numpy as np
import wfdb
import ast
import sklearn
import heartpy as hp

import tensorflow as tf
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, Input
from tensorflow.keras.models import Sequential
from sklearn.preprocessing import MultiLabelBinarizer

C:\Users\Bloby\PycharmProjects\ECG_Classification_Perturbation\.venv\Lib\site-packages\heartpy\datautils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


In [2]:
def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

In [3]:
path = 'ptb-xl/'
sampling_rate=500

# load and convert annotation data
Y = pd.read_csv(path+'ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load raw signal data
X = load_raw_data(Y, sampling_rate, path)

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv(path+'scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)



In [4]:
# Filter stuff
## ECG data typically has low pass filters set to 150 Hz. It helps smooth out the ECG signal but may impact the final amplitudes
Low_cutoff = 150.0

## High pass filters are often 0.5 Hz to cancel "respiration arifacts"
High_cutoff = 0.5

## we could also have power line filters to reduce noise caused by external electronics or anti-aliasing filters to reduce errors when the nyquist frequency is violated, but we are going to assume the data collected was done so expertly. If patients and rooms are prepared properly no filters may be needed but in some cases it is required.
# og_X = np.copy(X)
# for x_ind in range(1):
#     for index in range(1):
#         print(np.transpose(X[x_ind, :,index]))
#         X[x_ind, :, index] = hp.filter_signal(data = np.transpose(X[x_ind, :,index]), cutoff=[High_cutoff, Low_cutoff], sample_rate=sampling_rate, filtertype='bandpass')
#         print(np.transpose(X[x_ind, :, index]))


filt_X = np.copy(X)
for x_ind in range(len(filt_X)):
    for index in range(len(filt_X[x_ind, 0, :])):
        filt_X[x_ind, :, index] = hp.filter_signal(data = np.transpose(X[x_ind, :,index]), cutoff=[High_cutoff, Low_cutoff], sample_rate=sampling_rate, filtertype='bandpass')

In [5]:
np.array_equal(X, filt_X)

False

In [6]:
# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y.strat_fold != test_fold)]
X_filt_train = filt_X[np.where(Y.strat_fold != test_fold)]
y_train = Y[(Y.strat_fold != test_fold)].diagnostic_superclass

# Test
X_test = X[np.where(Y.strat_fold == test_fold)]
X_filt_test = filt_X[np.where(Y.strat_fold == test_fold)]
y_test = Y[Y.strat_fold == test_fold].diagnostic_superclass

### Fixing Y data for proper training format

In [7]:
mlb = MultiLabelBinarizer()
y_train_enc = mlb.fit_transform(y_train)
y_test_enc = mlb.fit_transform(y_test)
print("Encoded Labels (y_encoded):\n", y_train_enc)
print("\nClass Names (Order of Columns):\n", mlb.classes_)

Encoded Labels (y_encoded):
 [[0 0 0 1 0]
 [0 0 0 1 0]
 [0 0 0 1 0]
 ...
 [0 0 0 0 1]
 [0 0 0 1 0]
 [0 0 0 1 0]]

Class Names (Order of Columns):
 ['CD' 'HYP' 'MI' 'NORM' 'STTC']


# Model Training

In [43]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Early stopping - stop if validation loss doesn't improve
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

# Reduce learning rate when validation loss plateaus
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=0.00001,
    verbose=1
)

# Save best model
checkpoint = ModelCheckpoint(
    'best_ecg_modelv2.keras',
    monitor='val_auc',
    mode='max',
    save_best_only=True,
    verbose=1
)

callbacks = [early_stop, reduce_lr, checkpoint]

## Model structure

In [44]:
# THis is the number of classes. For Superclass it will be 5, if we do all of or a subset of the subclasses there can be more or less.
NUM_CLASSES = 5

# The input shape for the dataset
IN_SHAPE = (sampling_rate*10,12)

model = Sequential([
    Input(IN_SHAPE),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=32, kernel_size=2, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='sigmoid')]
)

## Compiling and Fitting (No filtering)

In [45]:
model.compile(
    optimizer='adam', 
    loss='binary_crossentropy', # could also use categorical_crossentropy here for a single choice per input. Change output to softmax if doing that approach though
    metrics=['accuracy', tf.keras.metrics.AUC(multi_label=True)]
)

In [15]:
model.fit(X_train, y_train_enc, epochs=20, verbose=1, validation_data=(X_test, y_test_enc))

Epoch 1/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 46s 73ms/step - accuracy: 0.4991 - auc: 0.7565 - loss: 0.4595 - val_accuracy: 0.5601 - val_auc: 0.8139 - val_loss: 0.4236
Epoch 2/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 43s 70ms/step - accuracy: 0.5800 - auc: 0.8346 - loss: 0.3930 - val_accuracy: 0.6087 - val_auc: 0.8449 - val_loss: 0.3826
Epoch 3/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 43s 70ms/step - accuracy: 0.6102 - auc: 0.8617 - loss: 0.3627 - val_accuracy: 0.5901 - val_auc: 0.8520 - val_loss: 0.3825
Epoch 4/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 43s 70ms/step - accuracy: 0.6290 - auc: 0.8804 - loss: 0.3407 - val_accuracy: 0.6156 - val_auc: 0.8515 - val_loss: 0.3825
Epoch 5/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 43s 70ms/step - accuracy: 0.6486 - auc: 0.8941 - loss: 0.3223 - val_accuracy: 0.5983 - val_auc: 0.8539 - val_loss: 0.3742
Epoch 6/20
613/613 ━━━━━━━━━━━━━━━━━━━━ 43s 70ms/step - accuracy: 0.6648 - auc: 0.9090 - loss: 0.2981 - val_accuracy: 0.6101 - val_auc: 0.8458 - val_loss: 0.4069
Epoch 7/20
613/613 ━━━━━━━━━

## Compiling and Fitting (filtering)

In [48]:
model2 = Sequential([
    Input(IN_SHAPE),
    Conv1D(filters=32, kernel_size=7, activation='relu'),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=8),
    Conv1D(filters=128, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=32),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.4),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation='sigmoid')]
)
optimizer = Adam(learning_rate=0.002)
model2.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    # could also use categorical_crossentropy here for a single choice per input. Change output to softmax if doing that approach though
    metrics=['accuracy', tf.keras.metrics.AUC(multi_label=True, name='auc')]
)

In [49]:
model2.fit(X_filt_train, y_train_enc, epochs=25, verbose=1, validation_data=(X_filt_test, y_test_enc), callbacks=callbacks)

Epoch 1/25
612/613 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.4733 - auc: 0.7092 - loss: 0.4901
Epoch 1: val_auc did not improve from 0.88160
613/613 ━━━━━━━━━━━━━━━━━━━━ 20s 29ms/step - accuracy: 0.5469 - auc: 0.7880 - loss: 0.4321 - val_accuracy: 0.6146 - val_auc: 0.8603 - val_loss: 0.3676 - learning_rate: 0.0020
Epoch 2/25
612/613 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6230 - auc: 0.8554 - loss: 0.3634
Epoch 2: val_auc improved from 0.88160 to 0.88939, saving model to best_ecg_modelv2.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.6186 - auc: 0.8582 - loss: 0.3607 - val_accuracy: 0.6351 - val_auc: 0.8894 - val_loss: 0.3336 - learning_rate: 0.0020
Epoch 3/25
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.6355 - auc: 0.8703 - loss: 0.3457
Epoch 3: val_auc did not improve from 0.88939
613/613 ━━━━━━━━━━━━━━━━━━━━ 16s 26ms/step - accuracy: 0.6402 - auc: 0.8724 - loss: 0.3423 - val_accuracy: 0.6410 - val_auc: 0.8851 - val_loss: 0.3370 - learning_rat

## Testing

In [50]:
loss, accuracy, auc_score = model.evaluate(X_test, y_test_enc, verbose=1)

# Print the results
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test AUC: {auc_score:.4f}")

loss2, accuracy2, auc_score2 = model2.evaluate(X_filt_test, y_test_enc, verbose=1)

# Print the results
print(f"Test Loss: {loss2:.4f}")
print(f"Test Accuracy: {accuracy2:.4f}")
print(f"Test AUC: {auc_score2:.4f}")

69/69 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.2025 - auc_6: 0.4833 - loss: 0.6907
Test Loss: 0.6907
Test Accuracy: 0.2025
Test AUC: 0.4833
69/69 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.6706 - auc: 0.9079 - loss: 0.2946
Test Loss: 0.2946
Test Accuracy: 0.6706
Test AUC: 0.9079


In [8]:
from tensorflow.keras.layers import BatchNormalization, Dropout, Input, Add
from tensorflow.keras.models import Model

# Improved model with Batch Normalization and Dropout
NUM_CLASSES = 5
IN_SHAPE = (5000, 12)

def create_improved_cnn():
    inputs = Input(shape=IN_SHAPE)
    
    # First Conv Block
    x = Conv1D(filters=64, kernel_size=7, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)
    
    # Second Conv Block
    x = Conv1D(filters=128, kernel_size=5, padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)
    
    # Third Conv Block
    x = Conv1D(filters=256, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.4)(x)
    
    # Fourth Conv Block
    x = Conv1D(filters=256, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.4)(x)
    
    # Global Average Pooling (better than Flatten)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    
    # Dense layers
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.5)(x)
    
    # Output layer
    outputs = Dense(NUM_CLASSES, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Create the improved model
model_v2 = create_improved_cnn()

In [10]:
from tensorflow.keras.optimizers import Adam
# Use a lower learning rate for better convergence
optimizer = Adam(learning_rate=0.005)

model_v2.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(multi_label=True, name='auc')]
)


In [11]:


print("Starting training with improved model...")
print("="*60)

history = model_v2.fit(
    X_filt_train, 
    y_train_enc,
    validation_data=(X_filt_test, y_test_enc),
    epochs=50,  # Will stop early if no improvement
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)


Starting training with improved model...
Epoch 1/50
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 293ms/step - accuracy: 0.4985 - auc: 0.7515 - loss: 0.4821
Epoch 1: val_auc improved from None to 0.86084, saving model to best_ecg_model.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 187s 299ms/step - accuracy: 0.5417 - auc: 0.7975 - loss: 0.4308 - val_accuracy: 0.5933 - val_auc: 0.8608 - val_loss: 0.3682 - learning_rate: 0.0050
Epoch 2/50
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 279ms/step - accuracy: 0.5914 - auc: 0.8450 - loss: 0.3831
Epoch 2: val_auc improved from 0.86084 to 0.87945, saving model to best_ecg_model.keras
613/613 ━━━━━━━━━━━━━━━━━━━━ 174s 284ms/step - accuracy: 0.6024 - auc: 0.8493 - loss: 0.3779 - val_accuracy: 0.6388 - val_auc: 0.8794 - val_loss: 0.3488 - learning_rate: 0.0050
Epoch 3/50
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 283ms/step - accuracy: 0.6182 - auc: 0.8603 - loss: 0.3660
Epoch 3: val_auc did not improve from 0.87945
613/613 ━━━━━━━━━━━━━━━━━━━━ 176s 288ms/step - accuracy: 0.6244 - auc: 0.8641 - 

In [12]:
loss, accuracy, auc_score = model_v2.evaluate(X_test, y_test_enc, verbose=1)

print(f"\n RESULTS:")
print(f"   Test Loss: {loss:.4f}")
print(f"   Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   Test AUC: {auc_score:.4f}")


69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 45ms/step - accuracy: 0.6656 - auc: 0.8913 - loss: 0.3313

 RESULTS:
   Test Loss: 0.3313
   Test Accuracy: 0.6656 (66.56%)
   Test AUC: 0.8913


In [25]:
NUM_CLASSES = 5
IN_SHAPE = (5000, 12)

def create_improved_cnn_V2():
    inputs = Input(shape=IN_SHAPE)
    
    # First Conv Block
    x = Conv1D(filters=64, kernel_size=9, padding='same')(inputs)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)
    
    # Second Conv Block
    x = Conv1D(filters=128, kernel_size=7, padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.3)(x)
    
    # Third Conv Block
    x = Conv1D(filters=256, kernel_size=3, padding='same')(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation('relu')(x)
    x = MaxPooling1D(pool_size=2)(x)
    x = Dropout(0.4)(x)
    
    # Global Average Pooling (better than Flatten)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    
    # Dense layers
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.5)(x)
    
    # Output layer
    outputs = Dense(NUM_CLASSES, activation='sigmoid')(x)
    
    model = Model(inputs=inputs, outputs=outputs)
    return model

# Create the improved model
model_v3 = create_improved_cnn()

In [26]:
optimizer = Adam(learning_rate=0.005)

model_v3.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(multi_label=True, name='auc')]
)


In [27]:
history = model_v3.fit(
    X_filt_train, 
    y_train_enc,
    validation_data=(X_filt_test, y_test_enc),
    epochs=50,  # Will stop early if no improvement
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/50
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step - accuracy: 0.5221 - auc: 0.7687 - loss: 0.4633
Epoch 1: val_auc did not improve from 0.92070
613/613 ━━━━━━━━━━━━━━━━━━━━ 175s 282ms/step - accuracy: 0.5655 - auc: 0.8126 - loss: 0.4179 - val_accuracy: 0.6028 - val_auc: 0.8604 - val_loss: 0.3789 - learning_rate: 0.0050
Epoch 2/50
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 277ms/step - accuracy: 0.5998 - auc: 0.8473 - loss: 0.3814
Epoch 2: val_auc did not improve from 0.92070
613/613 ━━━━━━━━━━━━━━━━━━━━ 172s 281ms/step - accuracy: 0.6134 - auc: 0.8521 - loss: 0.3736 - val_accuracy: 0.6397 - val_auc: 0.8795 - val_loss: 0.3430 - learning_rate: 0.0050
Epoch 3/50
613/613 ━━━━━━━━━━━━━━━━━━━━ 0s 275ms/step - accuracy: 0.6359 - auc: 0.8634 - loss: 0.3586
Epoch 3: val_auc did not improve from 0.92070
613/613 ━━━━━━━━━━━━━━━━━━━━ 171s 279ms/step - accuracy: 0.6329 - auc: 0.8660 - loss: 0.3565 - val_accuracy: 0.6369 - val_auc: 0.8874 - val_loss: 0.3507 - learning_rate: 0.0050
Epoch 4/50
613/613 ━━━━━━

In [31]:
loss, accuracy, auc_score = model_v3.evaluate(X_filt_test, y_test_enc, verbose=1)

print(f"\n RESULTS:")
print(f"   Test Loss: {loss:.4f}")
print(f"   Test Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   Test AUC: {auc_score:.4f}")

69/69 ━━━━━━━━━━━━━━━━━━━━ 3s 41ms/step - accuracy: 0.6028 - auc: 0.8604 - loss: 0.3789

 RESULTS:
   Test Loss: 0.3789
   Test Accuracy: 0.6028 (60.28%)
   Test AUC: 0.8604
